<a href="https://colab.research.google.com/github/aditi19sahu/eRes/blob/main/eRes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Base-year calibration data
- **Road-fuel use by vehicle segment.** Road transport burns about half of India's ≈5.6 mb/d of oil. The split across two-wheelers, three-wheelers, cars, buses and trucks is the author's estimate, consistent with petrol and diesel consumption patterns.
- **Electricity needed per litre displaced.** An EV doesn't remove oil for free; it adds grid demand. The conversion follows from comparing fuel economy and electric efficiency. For example, a petrol two-wheeler does about 50 km/L, while an electric one uses about 25 Wh/km. So one litre of petrol displaced needs about 50 × 0.025 ≈ 1.3 kWh. A diesel truck at about 4 km/L against about 1.2 kWh/km gives about 4.8 kWh per litre.


Model parameters


1. **Capacity factors (CF).** Annual generation = capacity × CF × 8,760 hours.
   - Solar's 18.5% and wind's 23% are back-solved from 2025 data.
   - Wind's CF rises to 30% by 2047 as taller turbines reach steadier winds.
   - Nuclear's 80% reflects a well-run reactor fleet.
2. **Growth rates.** Electricity demand, road and non-road oil demand, and real GDP. All compound annually: $X_t = X_{t-1}(1+g)$.
3. **Emission and fuel factors.**
   - Coal emits about 0.98 tCO₂ per MWh at India's fleet efficiency; gas about 0.45.
   - Indian coal plants burn about 0.63 kg of coal per kWh (calibrated to 2025).
4. **Costs, in real 2025 ₹/kWh.**
   - *Clean sources* use a **levelised cost** (LCOE): lifetime cost ÷ lifetime output. It declines each year at a learning rate, down to a floor.
   - *Coal* is split into a **variable** (fuel) part and a **fixed** (capital) part. That split is the heart of the report's cost-structure argument (Lens A): fixed costs per kWh rise when plants run less.
   - *Storage* is priced as an **adder** per kWh time-shifted (Ember: ₹4.0–4.5).
   - *Nuclear* has separate costs for the existing and new fleet.
5. **Storage threshold.** The share of demand that solar can serve directly in daylight before its output must be stored (20%). Round-trip efficiency is 90%.
   - **The threshold is a calibration, not a physical law.** It was chosen so the current-policy scenario's 2032 storage need roughly matches CEA's National Electricity Plan (≈236 GWh of batteries plus ≈175 GWh of pumped hydro).

# India Electrotech Sovereignty — Scenario Model

Annual model (2025–2047) of India's electricity mix, costs, emissions and oil import bill under three policy scenarios plus a custom one. An accounting model, not a forecast or hourly grid model. Run all cells in order.

## 1. Global assumptions
Prices, demand growth and technology costs shared by every scenario.

In [ ]:
#@title Global assumptions { display-mode: "form" }
oil_price_base = 80  #@param {type:"slider", min:50, max:160, step:5}
oil_price_stress = 110  #@param {type:"slider", min:50, max:160, step:5}
demand_growth_2025_35_pct = 5.5  #@param {type:"slider", min:3.0, max:8.0, step:0.5}
demand_growth_2035_47_pct = 4.5  #@param {type:"slider", min:2.0, max:7.0, step:0.5}
solar_capacity_factor_pct = 18.5  #@param {type:"slider", min:15.0, max:23.0, step:0.5}
solar_cost_decline_pct_per_yr = 1.5  #@param {type:"slider", min:0.0, max:5.0, step:0.5}
storage_cost_decline_pct_per_yr = 4.0  #@param {type:"slider", min:0.0, max:10.0, step:0.5}
nuclear_new_build_cost_inr_per_kwh = 5.75  #@param {type:"slider", min:4.0, max:9.0, step:0.25}

## 2. Custom scenario (S4)
Build rates, nuclear/hydro capacity and EV adoption for a user-defined scenario.

In [ ]:
#@title Custom scenario S4 { display-mode: "form" }
include_custom = True  #@param {type:"boolean"}
custom_label = "Custom"  #@param {type:"string"}
solar_add_2026_30_GW_per_yr = 60  #@param {type:"slider", min:0, max:200, step:5}
solar_add_2031_40_GW_per_yr = 70  #@param {type:"slider", min:0, max:200, step:5}
solar_add_2041_47_GW_per_yr = 80  #@param {type:"slider", min:0, max:200, step:5}
wind_add_2026_30_GW_per_yr = 10  #@param {type:"slider", min:0, max:50, step:1}
wind_add_2031_47_GW_per_yr = 20  #@param {type:"slider", min:0, max:50, step:1}
nuclear_GW_2035 = 25  #@param {type:"slider", min:9, max:60, step:1}
nuclear_GW_2047 = 50  #@param {type:"slider", min:9, max:150, step:1}
hydro_GW_2047 = 85  #@param {type:"slider", min:52, max:120, step:1}
#@markdown EV share of road activity in 2047 (0–1); 2035 = 2047 × ratio.
ev_2w_2047 = 0.8  #@param {type:"slider", min:0, max:1, step:0.05}
ev_3w_2047 = 0.95  #@param {type:"slider", min:0, max:1, step:0.05}
ev_cars_2047 = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
ev_buses_2047 = 0.8  #@param {type:"slider", min:0, max:1, step:0.05}
ev_trucks_2047 = 0.45  #@param {type:"slider", min:0, max:1, step:0.05}
ev_2035_as_fraction_of_2047 = 0.45  #@param {type:"slider", min:0.1, max:1, step:0.05}

## 3. Model

### 3.1 Base-year data (2025)
Observed 2025 generation, capacity and oil use used for calibration.

In [ ]:
import copy

YEARS = list(range(2025, 2048))

BASE = dict(
    gen_TWh=2081.6,
    coal=1474.15, gas=48.54, oil=4.10,
    nuclear=53.83, hydro=177.97, solar=195.99, wind=103.98, bio=23.04,
    cap_solar=140.0, cap_wind=54.0,
    cap_nuc=8.8, cap_hydro=52.0,
    road_oil_mbd=2.70,
    nonroad_oil_mbd=2.90,
    dom_crude_mbd=0.55,
    gdp_usd_T=4.0,
    pop_bn=1.46,
)

SEG = dict(two_w=0.55, three_w=0.05, cars=0.55, buses=0.25, trucks=1.30)

KWH_PER_L = dict(two_w=1.3, three_w=1.8, cars=2.3, buses=4.5, trucks=4.8)

### 3.2 Parameters
Capacity factors, growth rates, costs and emission factors.

In [ ]:
PARAMS = dict(
    cf_solar=0.185, cf_wind_2025=0.23, cf_wind_2047=0.30, cf_nuc=0.80, cf_hydro=0.38,
    bio_TWh=25, gas_TWh=45, oil_TWh=4,

    demand_growth_2025_35=0.055, demand_growth_2035_47=0.045,
    road_oil_growth=0.035, nonroad_oil_growth=0.030,
    gdp_usd_growth_2025_35=0.065, gdp_usd_growth_2035_47=0.055,
    pop_2035=1.55, pop_2047=1.62,

    co2_coal=0.98, co2_gas=0.45,
    coal_kg_per_kWh=0.63,

    lcoe_solar_2025=2.6, solar_decline=0.015, solar_floor=2.0,
    lcoe_wind_2025=3.4, wind_decline=0.01, wind_floor=2.8,
    storage_adder_2025=4.25, storage_decline=0.04, storage_floor=2.0,
    coal_var=2.9,
    coal_fixed_at_69=1.9,
    nuc_existing=3.8, nuc_new=6.0,
    hydro=4.0, gas=8.0, bio=6.0,

    direct_solar_absorb=0.20,
    rte=0.90,
)

### 3.3 Scenarios
S1 Stalled, S2 Current policy, S3 Electrotech sovereignty.

In [ ]:
SCEN = {
    "S1_Stalled": dict(
        label="Stalled",
        solar_add=[(2026, 2030, 30), (2031, 2047, 30)],
        wind_add=[(2026, 2047, 5)],
        nuc_path={2025: 8.8, 2032: 15, 2035: 18, 2047: 30},
        hydro_path={2025: 52, 2035: 62, 2047: 75},
        coal_cap={2025: 240, 2032: 275, 2047: 275},
        ev={"two_w": (0.15, 0.40), "three_w": (0.40, 0.70), "cars": (0.05, 0.25),
            "buses": (0.10, 0.40), "trucks": (0.02, 0.15)}),
    "S2_Policy": dict(
        label="Current policy",
        solar_add=[(2026, 2030, 45), (2031, 2047, 50)],
        wind_add=[(2026, 2030, 8), (2031, 2047, 12)],
        nuc_path={2025: 8.8, 2032: 22, 2035: 28, 2047: 60},
        hydro_path={2025: 52, 2035: 70, 2047: 85},
        coal_cap={2025: 240, 2032: 275, 2047: 250},
        ev={"two_w": (0.25, 0.70), "three_w": (0.55, 0.90), "cars": (0.10, 0.50),
            "buses": (0.20, 0.70), "trucks": (0.05, 0.35)}),
    "S3_Electrotech": dict(
        label="Electrotech",
        solar_add=[(2026, 2030, 60), (2031, 2040, 85), (2041, 2047, 115)],
        wind_add=[(2026, 2030, 12), (2031, 2040, 20), (2041, 2047, 28)],
        nuc_path={2025: 8.8, 2032: 22.5, 2035: 32, 2047: 100},
        hydro_path={2025: 52, 2035: 75, 2047: 95},
        coal_cap={2025: 240, 2032: 275, 2047: 200},
        ev={"two_w": (0.40, 0.90), "three_w": (0.70, 0.98), "cars": (0.18, 0.75),
            "buses": (0.35, 0.90), "trucks": (0.10, 0.60)}),
}

### 3.4 Helpers
Interpolation of anchor years, build schedules and EV adoption ramps.

In [ ]:
def interp(path, y):
    ks = sorted(path)
    if y <= ks[0]:
        return path[ks[0]]
    for a, b in zip(ks, ks[1:]):
        if a <= y <= b:
            return path[a] + (path[b] - path[a]) * (y - a) / (b - a)
    return path[ks[-1]]

def adds(schedule, y):
    for first, last, gw in schedule:
        if first <= y <= last:
            return gw
    return 0

def ev_share(pair, y):
    s35, s47 = pair
    if y <= 2025:
        return 0.0
    if y <= 2035:
        return s35 * (y - 2025) / 10
    return s35 + (s47 - s35) * (y - 2035) / 12

### 3.5 Demand and transport electrification
Grows underlying demand and adds EV charging load; tracks oil displaced.

In [ ]:
def advance_exogenous(state, sc, P, y):
    if y == 2025:
        return
    g = P["demand_growth_2025_35"] if y <= 2035 else P["demand_growth_2035_47"]
    state["underlying_TWh"] *= 1 + g
    state["solar_GW"] += adds(sc["solar_add"], y)
    state["wind_GW"] += adds(sc["wind_add"], y)
    for k in state["road_cf_mbd"]:
        state["road_cf_mbd"][k] *= 1 + P["road_oil_growth"]
    state["nonroad_mbd"] *= 1 + P["nonroad_oil_growth"]
    state["gdp_T"] *= 1 + (P["gdp_usd_growth_2025_35"] if y <= 2035 else P["gdp_usd_growth_2035_47"])

def electrify_transport(state, sc, y):
    displaced = {k: state["road_cf_mbd"][k] * ev_share(sc["ev"][k], y) for k in state["road_cf_mbd"]}
    disp_mbd = sum(displaced.values())
    ev_TWh = sum(displaced[k] * 1e6 * 159 * 365 * KWH_PER_L[k] for k in displaced) / 1e9
    return disp_mbd, ev_TWh

### 3.6 Clean generation
Generation = capacity × capacity factor × 8.76 for non-coal sources.

In [ ]:
def clean_generation(state, sc, P, y):
    nuc_GW, hyd_GW = interp(sc["nuc_path"], y), interp(sc["hydro_path"], y)
    if y == 2025:
        g = dict(solar=BASE["solar"], wind=BASE["wind"], nuclear=BASE["nuclear"], hydro=BASE["hydro"],
                 bio=BASE["bio"], gas=BASE["gas"], oil=BASE["oil"])
    else:
        cf_wind = P["cf_wind_2025"] + (P["cf_wind_2047"] - P["cf_wind_2025"]) * (y - 2025) / 22
        g = dict(solar=state["solar_GW"] * P["cf_solar"] * 8.76,
                 wind=state["wind_GW"] * cf_wind * 8.76,
                 nuclear=nuc_GW * P["cf_nuc"] * 8.76,
                 hydro=hyd_GW * P["cf_hydro"] * 8.76,
                 bio=P["bio_TWh"], gas=P["gas_TWh"], oil=P["oil_TWh"])
    g["clean"] = g["solar"] + g["wind"] + g["nuclear"] + g["hydro"] + g["bio"]
    return g, nuc_GW, hyd_GW

### 3.7 Coal residual
Coal meets remaining demand; new capacity is added if fleet utilisation would exceed 70%.

In [ ]:
def coal_residual(demand_TWh, g, sc, y):
    coal_TWh = max(demand_TWh - g["clean"] - g["gas"] - g["oil"], 0.0)
    curtail_TWh = max(g["clean"] + g["gas"] + g["oil"] - demand_TWh, 0.0)
    planned_GW = interp(sc["coal_cap"], y)
    needed_GW = coal_TWh / (8.76 * 0.70)
    coal_cap_GW = max(planned_GW, needed_GW)
    extra_coal_GW = coal_cap_GW - planned_GW
    plf = coal_TWh / (coal_cap_GW * 8.76)
    return coal_TWh, curtail_TWh, coal_cap_GW, extra_coal_GW, plf

### 3.8 Storage need
Solar beyond the directly absorbed share is time-shifted through storage.

In [ ]:
def storage_need(solar_TWh, demand_TWh, P):
    solar_share = solar_TWh / demand_TWh
    shifted_TWh = max(0.0, solar_share - P["direct_solar_absorb"]) * demand_TWh
    storage_GWh_per_day = shifted_TWh * 1000 / 365 / P["rte"]
    return shifted_TWh, storage_GWh_per_day

### 3.9 Costs
Unit costs per source and the system's average generation cost.

In [ ]:
def unit_costs(P, y, coal_TWh, plf, extra_coal_GW, coal_cap_GW, nuc_GW):
    t = y - 2025
    c = dict(
        solar=max(P["lcoe_solar_2025"] * (1 - P["solar_decline"]) ** t, P["solar_floor"]),
        wind=max(P["lcoe_wind_2025"] * (1 - P["wind_decline"]) ** t, P["wind_floor"]),
        storage=max(P["storage_adder_2025"] * (1 - P["storage_decline"]) ** t, P["storage_floor"]),
        hydro=P["hydro"], bio=P["bio"], gas=P["gas"],
    )
    c["coal"] = P["coal_var"] + P["coal_fixed_at_69"] * 0.69 / max(plf, 0.2) if coal_TWh > 0 else 0
    if coal_TWh > 0 and extra_coal_GW > 0:
        existing_at_70 = P["coal_var"] + P["coal_fixed_at_69"] * 0.69 / 0.70
        c["coal"] += (extra_coal_GW / coal_cap_GW) * (6.0 - existing_at_70)
    new_frac = max(nuc_GW - 8.8, 0) / nuc_GW
    c["nuclear"] = P["nuc_existing"] * (1 - new_frac) + P["nuc_new"] * new_frac
    return c

def average_cost(g, coal_TWh, shifted_TWh, c, demand_TWh):
    total = (g["solar"] * c["solar"] + g["wind"] * c["wind"] + g["nuclear"] * c["nuclear"]
             + g["hydro"] * c["hydro"] + g["bio"] * c["bio"] + g["gas"] * c["gas"]
             + coal_TWh * c["coal"] + shifted_TWh * c["storage"])
    return total / demand_TWh

### 3.10 Emissions and oil import bill

In [ ]:
def emissions(g, coal_TWh, P):
    co2_Mt = coal_TWh * P["co2_coal"] + g["gas"] * P["co2_gas"]
    coal_Mt = coal_TWh * P["coal_kg_per_kWh"]
    return co2_Mt, coal_Mt

def oil_import_bill(state, disp_mbd, oil_price):
    oil_demand_mbd = sum(state["road_cf_mbd"].values()) - disp_mbd + state["nonroad_mbd"]
    bill_bn = (oil_demand_mbd - BASE["dom_crude_mbd"]) * 365 * oil_price / 1000
    pct_gdp = bill_bn / (state["gdp_T"] * 1000) * 100
    return oil_demand_mbd, bill_bn, pct_gdp

### 3.11 Investment
Annual capital expenditure for new capacity.

In [ ]:
INR_PER_USD = 88

def capex_increment(now, prev, y):
    t = y - 2025
    solar_cost = max(3.8 * 0.98 ** t, 2.8)
    storage_cost = max(1.4 * 0.95 ** t, 0.6)
    inr_crore = ((now["solar_GW"] - prev["solar_GW"]) * 1000 * solar_cost
                 + (now["wind_GW"] - prev["wind_GW"]) * 1000 * 6.5
                 + (now["nuc_GW"] - prev["nuc_GW"]) * 1000 * 18
                 + (now["hydro_GW"] - prev["hydro_GW"]) * 1000 * 12
                 + max(now["storage_GWh"] - prev["storage_GWh"], 0) * 1000 * storage_cost
                 + max(now["coal_cap_GW"] - prev["coal_cap_GW"], 0) * 1000 * 9)
    return inr_crore * 1e7 / INR_PER_USD / 1e9

### 3.12 Simulation loop
Runs the modules year by year for one scenario.

In [ ]:
def run(sc, P, oil_price=80.0):
    state = dict(underlying_TWh=BASE["gen_TWh"], solar_GW=BASE["cap_solar"], wind_GW=BASE["cap_wind"],
                 road_cf_mbd=dict(SEG), nonroad_mbd=BASE["nonroad_oil_mbd"], gdp_T=BASE["gdp_usd_T"])
    out, prev, capex_cum_bn = {}, None, 0.0
    for y in YEARS:
        advance_exogenous(state, sc, P, y)
        disp_mbd, ev_TWh = electrify_transport(state, sc, y)
        demand_TWh = state["underlying_TWh"] + ev_TWh
        g, nuc_GW, hyd_GW = clean_generation(state, sc, P, y)
        coal_TWh, curtail_TWh, coal_cap_GW, extra_coal_GW, plf = coal_residual(demand_TWh, g, sc, y)
        shifted_TWh, storage_GWh = storage_need(g["solar"], demand_TWh, P)
        c = unit_costs(P, y, coal_TWh, plf, extra_coal_GW, coal_cap_GW, nuc_GW)
        avg_cost = average_cost(g, coal_TWh, shifted_TWh, c, demand_TWh)
        co2_Mt, coal_Mt = emissions(g, coal_TWh, P)
        oil_demand_mbd, bill_bn, bill_pct = oil_import_bill(state, disp_mbd, oil_price)
        pop_bn = interp({2025: BASE["pop_bn"], 2035: P["pop_2035"], 2047: P["pop_2047"]}, y)

        now = dict(
            demand_TWh=demand_TWh,
            ev_TWh=ev_TWh,
            pc_kWh=demand_TWh / pop_bn,
            solar_GW=state["solar_GW"], wind_GW=state["wind_GW"], nuc_GW=nuc_GW, hydro_GW=hyd_GW,
            coal_cap_GW=coal_cap_GW,
            extra_coal_GW=extra_coal_GW,
            coal_TWh=coal_TWh,
            coal_share=coal_TWh / demand_TWh,
            clean_share=(g["clean"] - curtail_TWh) / demand_TWh,
            coal_plf=plf,
            curtail_TWh=curtail_TWh,
            storage_GWh=storage_GWh,
            avg_cost_INR=avg_cost,
            power_CO2_Mt=co2_Mt, coal_Mt=coal_Mt,
            oil_displaced_mbd=disp_mbd, oil_demand_mbd=oil_demand_mbd,
            oil_bill_bn=bill_bn, oil_bill_pct_gdp=bill_pct, gdp_T=state["gdp_T"],
        )
        if prev is not None:
            capex_cum_bn += capex_increment(now, prev, y)
        now["capex_cum_bn"] = capex_cum_bn
        out[y] = now
        prev = now
    return out

print("Model loaded: modules §3.5–§3.11 and run() are defined.")

Model loaded: modules §3.5–§3.11 and run() are defined.


## 4. Run all scenarios
Applies the form settings and runs each scenario at base and stress oil prices.

In [ ]:
P_DEFAULT = copy.deepcopy(PARAMS)
P = copy.deepcopy(PARAMS)
P.update(
    demand_growth_2025_35=demand_growth_2025_35_pct / 100,
    demand_growth_2035_47=demand_growth_2035_47_pct / 100,
    cf_solar=solar_capacity_factor_pct / 100,
    solar_decline=solar_cost_decline_pct_per_yr / 100,
    storage_decline=storage_cost_decline_pct_per_yr / 100,
    nuc_new=nuclear_new_build_cost_inr_per_kwh,
)

scenarios = copy.deepcopy(SCEN)
if include_custom:
    r = ev_2035_as_fraction_of_2047
    scenarios["S4_Custom"] = dict(
        label=custom_label or "Custom",
        solar_add=[(2026, 2030, solar_add_2026_30_GW_per_yr), (2031, 2040, solar_add_2031_40_GW_per_yr),
                   (2041, 2047, solar_add_2041_47_GW_per_yr)],
        wind_add=[(2026, 2030, wind_add_2026_30_GW_per_yr), (2031, 2047, wind_add_2031_47_GW_per_yr)],
        nuc_path={2025: 8.8, 2035: max(nuclear_GW_2035, 8.8), 2047: max(nuclear_GW_2047, 8.8)},
        hydro_path={2025: 52, 2047: hydro_GW_2047},
        coal_cap={2025: 240, 2032: 275, 2047: 250},
        ev={"two_w": (ev_2w_2047 * r, ev_2w_2047), "three_w": (ev_3w_2047 * r, ev_3w_2047),
            "cars": (ev_cars_2047 * r, ev_cars_2047), "buses": (ev_buses_2047 * r, ev_buses_2047),
            "trucks": (ev_trucks_2047 * r, ev_trucks_2047)},
    )

results = {k: run(sc, P, oil_price_base) for k, sc in scenarios.items()}
stress = {k: run(sc, P, oil_price_stress) for k, sc in scenarios.items()}
print("Ran:", ", ".join(sc["label"] for sc in scenarios.values()))

Ran: Stalled, Current policy, Electrotech, Custom


## 5. Self-check
Confirms default parameters reproduce the reference results.

In [ ]:
ref3 = run(SCEN["S3_Electrotech"], P_DEFAULT, 80)[2047]
ref1 = run(SCEN["S1_Stalled"], P_DEFAULT, 80)[2047]
checks = [
    ("S3 2047 coal generation (TWh)", ref3["coal_TWh"], 1000, 5),
    ("S3 2047 clean share", ref3["clean_share"], 0.846, 0.005),
    ("S3 2047 avg generation cost (INR/kWh)", ref3["avg_cost_INR"], 3.76, 0.01),
    ("S1 2047 new coal beyond pipeline (GW)", ref1["extra_coal_GW"], 381, 2),
]
for name, got, want, tol in checks:
    print(f"{'OK  ' if abs(got - want) <= tol else 'DIFF'}  {name}: model {got:,.3f} vs report {want:,}")
print("\nYour settings match the report's defaults." if P == P_DEFAULT
      else "\nYour global assumptions differ from the report's, so the core-scenario results below will differ too.")

OK    S3 2047 coal generation (TWh): model 999.793 vs report 1,000
OK    S3 2047 clean share: model 0.846 vs report 0.846
OK    S3 2047 avg generation cost (INR/kWh): model 3.757 vs report 3.76
OK    S1 2047 new coal beyond pipeline (GW): model 380.964 vs report 381

Your settings match the report's defaults.


## 6. Results tables

In [ ]:
import pandas as pd
from IPython.display import display

ROWS = [
    ("Electricity demand incl. EVs (TWh)", "demand_TWh", "{:,.0f}"),
    ("  of which EV charging (TWh)", "ev_TWh", "{:,.0f}"),
    ("Per-capita consumption (kWh)", "pc_kWh", "{:,.0f}"),
    ("Solar (GW)", "solar_GW", "{:,.0f}"),
    ("Wind (GW)", "wind_GW", "{:,.0f}"),
    ("Nuclear (GW)", "nuc_GW", "{:,.0f}"),
    ("Coal capacity required (GW)", "coal_cap_GW", "{:,.0f}"),
    ("  new coal beyond pipeline (GW)", "extra_coal_GW", "{:,.0f}"),
    ("Coal generation (TWh)", "coal_TWh", "{:,.0f}"),
    ("Coal share of generation", "coal_share", "{:.0%}"),
    ("Clean share of generation", "clean_share", "{:.0%}"),
    ("Storage throughput need (GWh/day)", "storage_GWh", "{:,.0f}"),
    ("Avg generation cost (2025 ₹/kWh)", "avg_cost_INR", "{:.2f}"),
    ("Power-sector CO₂ (Mt)", "power_CO2_Mt", "{:,.0f}"),
    ("Oil displaced by EVs (mb/d)", "oil_displaced_mbd", "{:.2f}"),
    ("Net oil import bill, base price ($bn)", "oil_bill_bn", "{:,.0f}"),
    ("  as % of GDP, base price", "oil_bill_pct_gdp", "{:.1f}%"),
    ("Cumulative capex from 2026 ($bn)", "capex_cum_bn", "{:,.0f}"),
]

def results_table(years=(2025, 2035, 2047)):
    cols, data = [], {}
    for y in years:
        keys = ["S1_Stalled"] if y == 2025 else list(scenarios)
        for k in keys:
            col = "2025 (base)" if y == 2025 else f"{scenarios[k]['label']} {y}"
            cols.append(col)
            data[col] = ([fmt.format(results[k][y][key]) for _, key, fmt in ROWS]
                         + [f"{stress[k][y]['oil_bill_bn']:,.0f}", f"{stress[k][y]['oil_bill_pct_gdp']:.1f}%"])
    idx = [r[0] for r in ROWS] + [f"Net oil import bill, stress ${oil_price_stress}/bbl ($bn)", "  as % of GDP, stress price"]
    return pd.DataFrame(data, index=idx)[cols]

display(results_table())

,2025 (base),Stalled 2035,Current policy 2035,Electrotech 2035,Custom 2035,Stalled 2047,Current policy 2047,Electrotech 2047,Custom 2047
Electricity demand incl. EVs (TWh),"2,082","3,592","3,629","3,686","3,744","6,284","6,548","6,826","6,664"
of which EV charging (TWh),0,36,73,131,189,254,518,796,634
Per-capita consumption (kWh),"1,426","2,317","2,341","2,378","2,416","3,879","4,042","4,214","4,114"
Solar (GW),140,440,615,865,790,800,"1,215","2,095","1,700"
Wind (GW),54,104,154,214,204,164,298,510,444
Nuclear (GW),9,18,28,32,25,30,60,100,50
Coal capacity required (GW),240,364,290,260,270,656,492,200,332
new coal beyond pipeline (GW),0,89,20,0,0,381,242,0,82
Coal generation (TWh),"1,474","2,234","1,776","1,246","1,524","4,022","3,018","1,000","2,035"
Coal share of generation,71%,62%,49%,34%,41%,64%,46%,15%,31%


In [ ]:
#@title Cumulative outcomes, 2026–2047
cum = {}
for k, sc in scenarios.items():
    s = results[k]
    oil_mbbl = sum(s[y]["oil_displaced_mbd"] for y in YEARS[1:]) * 365
    cum[sc["label"]] = {
        "Cumulative coal generation (TWh)": f"{sum(s[y]['coal_TWh'] for y in YEARS[1:]):,.0f}",
        "Cumulative power-sector CO₂ (Gt)": f"{sum(s[y]['power_CO2_Mt'] for y in YEARS[1:]) / 1000:.1f}",
        "Oil displaced by EVs (bn bbl)": f"{oil_mbbl / 1000:.1f}",
        f"Value of displaced oil @ ${oil_price_base} ($bn)": f"{oil_mbbl * oil_price_base / 1000:,.0f}",
        f"Value of displaced oil @ ${oil_price_stress} ($bn)": f"{oil_mbbl * oil_price_stress / 1000:,.0f}",
        "Coal peak year": max(YEARS, key=lambda y: s[y]["coal_TWh"]),
    }
display(pd.DataFrame(cum))

,Stalled,Current policy,Electrotech,Custom
Cumulative coal generation (TWh),"55,143","43,680","26,464","35,105"
Cumulative power-sector CO₂ (Gt),54.5,43.3,26.4,34.8
Oil displaced by EVs (bn bbl),4.1,7.9,12.1,11.4
Value of displaced oil @ $80 ($bn),331,636,971,913
Value of displaced oil @ $110 ($bn),456,874,"1,335","1,255"
Coal peak year,2047,2047,2025,2047


## 7. Charts

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

COLORS = {"S1_Stalled": "#2a78d6", "S2_Policy": "#eb6834", "S3_Electrotech": "#1baf7a", "S4_Custom": "#eda100"}
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"

def _dodge(vals, min_gap):
    order = sorted(range(len(vals)), key=lambda i: vals[i])
    pos = list(vals)
    for a, b in zip(order, order[1:]):
        if pos[b] - pos[a] < min_gap:
            pos[b] = pos[a] + min_gap
    shift = (sum(pos) - sum(vals)) / len(vals)
    return [p - shift for p in pos]

def line_chart(metric, title, subtitle, ytitle, hoverfmt, labelfmt, data=None, from_zero=True, tickformat=None):
    data = data or results
    fig = go.Figure()
    series = []
    for k, sc in scenarios.items():
        ys = [data[k][y][metric] for y in YEARS]
        series.append((k, sc["label"], ys))
        fig.add_trace(go.Scatter(x=YEARS, y=ys, mode="lines", name=sc["label"],
                                 line=dict(color=COLORS[k], width=2),
                                 hovertemplate=f"{sc['label']}: %{{y:{hoverfmt}}}<extra></extra>"))
    lo = 0 if from_zero else min(min(ys) for _, _, ys in series)
    hi = max(max(ys) for _, _, ys in series)
    pad = (hi - lo) * 0.06
    lo_axis = 0 if from_zero else lo - pad
    label_y = _dodge([ys[-1] for _, _, ys in series], (hi + pad - lo_axis) * 0.07)
    for (k, label, ys), ly in zip(series, label_y):
        fig.add_annotation(x=YEARS[-1], y=ly, xanchor="left", xshift=10, showarrow=False,
                           text=f"<span style='color:{COLORS[k]}'>●</span> {label} {labelfmt.format(ys[-1])}",
                           font=dict(color=INK2, size=12))
    fig.update_layout(
        title=dict(text=f"{title}<br><span style='font-size:12px;color:{INK2}'>{subtitle}</span>",
                   font=dict(color=INK, size=16), x=0, xanchor="left", y=0.97, yanchor="top"),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=430, margin=dict(l=60, r=210, t=90, b=80),
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color=INK2, size=12),
        hovermode="x unified", hoverlabel=dict(bgcolor="white", font=dict(color=INK)),
        legend=dict(orientation="h", y=-0.14, yanchor="top", x=0, xanchor="left", font=dict(color=INK2)),
    )
    fig.update_xaxes(range=[2024.5, 2047.5], showgrid=False, linecolor=AXIS, tickcolor=AXIS, tickfont=dict(color=MUTED),
                     tickvals=[2025, 2030, 2035, 2040, 2045, 2047],
                     showspikes=True, spikecolor=AXIS, spikethickness=1, spikedash="solid", spikemode="across")
    fig.update_yaxes(title=dict(text=ytitle, font=dict(color=MUTED)), gridcolor=GRID, zeroline=False,
                     linecolor=AXIS, tickfont=dict(color=MUTED), range=[lo_axis, hi + pad], tickformat=tickformat)
    fig.show()

line_chart("coal_TWh", "Coal generation", "Coal fills whatever clean sources don't supply", "TWh per year", ",.0f", "{:,.0f} TWh")

In [ ]:
line_chart("clean_share", "Clean share of electricity generation", "Solar, wind, hydro, nuclear and biomass, net of curtailment",
           "Share of generation", ".0%", "{:.0%}", tickformat=".0%")

In [ ]:
line_chart("avg_cost_INR", "Average generation cost",
           "Real 2025 rupees per kWh, including storage for time-shifted solar. Note the transition hump before 2040",
           "₹ per kWh", ".2f", "₹{:.2f}", from_zero=False)

In [ ]:
line_chart("oil_bill_pct_gdp", f"Net oil import bill at ${oil_price_stress}/bbl",
           "Share of GDP. EV adoption bends the bill; non-road oil (LPG, petrochemicals, aviation) keeps it above zero",
           "% of GDP", ".2f", "{:.1f}%", data=stress)

In [ ]:
PARAMS = dict(
    # 1. Capacity factors (fraction of the year at full output)
    cf_solar=0.185, cf_wind_2025=0.23, cf_wind_2047=0.30, cf_nuc=0.80, cf_hydro=0.38,
    bio_TWh=25, gas_TWh=45, oil_TWh=4,            # small sources held flat after 2025, TWh/yr

    # 2. Growth rates (fraction per year, compounding)
    demand_growth_2025_35=0.055, demand_growth_2035_47=0.045,   # underlying electricity demand (before EVs)
    road_oil_growth=0.035, nonroad_oil_growth=0.030,            # oil demand if nothing electrified
    gdp_usd_growth_2025_35=0.065, gdp_usd_growth_2035_47=0.055, # real GDP
    pop_2035=1.55, pop_2047=1.62,                               # population anchors, billions

    # 3. Emission and fuel factors
    co2_coal=0.98, co2_gas=0.45,       # tonnes CO2 per MWh generated
    coal_kg_per_kWh=0.63,              # kg of coal burned per kWh (calibrated to 2025)

    # 4. Costs, real 2025 INR per kWh
    lcoe_solar_2025=2.6, solar_decline=0.015, solar_floor=2.0,          # solar LCOE, annual decline, floor
    lcoe_wind_2025=3.4, wind_decline=0.01, wind_floor=2.8,              # wind LCOE, annual decline, floor
    storage_adder_2025=4.25, storage_decline=0.04, storage_floor=2.0,   # cost per kWh time-shifted
    coal_var=2.9,                      # coal variable (fuel + O&M) cost per kWh
    coal_fixed_at_69=1.9,              # coal fixed (capital) cost per kWh *when the fleet runs at 69% PLF*
    nuc_existing=3.8, nuc_new=6.0,     # nuclear: existing fleet vs new build (CEA roadmap)
    hydro=4.0, gas=8.0, bio=6.0,       # other sources, flat

    # 5. Storage rule of thumb
    direct_solar_absorb=0.20,          # share of demand solar can serve directly before shifting is needed
    rte=0.90,                          # battery round-trip efficiency
)